# Training Table Walkthrough

Builds a training table from `pre_training_table.parquet`.

One row per tracking frame (≈10 Hz). All 121 `t.*` columns from the pre-training table pass through unchanged. Five columns are added:

| Column | Description |
|---|---|
| `p.event_label` | Event label inherited from the pre-training table |
| `data_split` | `train` / `validation` / `test` — assigned at the match level |
| `is_pass` | Binary target: 1 = PASS, 0 = anything else |
| `ball_speed_avg_xy` | Rolling mean of 2D frame-to-frame ball speed over ±5 frames (m / frame) |
| `closest_player_id` | ID of the visible player closest to the ball at this row's frame |
| `closest_player_team_id` | Team of the closest visible player at this row's frame |

## 1. Load data

In [220]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
from IPython.display import display

from driblab.config import CONFIG_PATH, MODEL_BASE_DATA_DIR
from driblab.features import match_splits

pd.set_option('display.max_columns', None)

PRE_TRAINING_PATH = MODEL_BASE_DATA_DIR / 'pre_training_table.parquet'

assert PRE_TRAINING_PATH.exists(), f'Missing {PRE_TRAINING_PATH} — run pre_training_table.ipynb first.'

df = pd.read_parquet(PRE_TRAINING_PATH)
t_cols = [c for c in df.columns if c.startswith('t.')]
p_cols = [c for c in df.columns if c.startswith('p.')]

print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
print(f'  t.* columns : {len(t_cols)}')
print(f'  p.* columns : {p_cols}')
print()
display(
    df[['t.match_id', 't.period', 't.frame', 't.Videotimestamp',
        'p.event_label', 't.ball_x', 't.ball_y']].head(6)
)

Loaded 1,986,630 rows, 122 columns
  t.* columns : 121
  p.* columns : ['p.event_label']



,t.match_id,t.period,t.frame,t.Videotimestamp,p.event_label,t.ball_x,t.ball_y
0,678949,1,0,1.0,PASS,NaN,NaN
1,678949,1,1,1.1,PASS,NaN,NaN
2,678949,1,2,1.2,PASS,NaN,NaN
3,678949,1,3,1.3,PASS,NaN,NaN
4,678949,1,4,1.4,PASS,NaN,NaN
5,678949,1,5,1.5,PASS,NaN,NaN


## 2. Add data_split column

Split assignments come from `config.yaml` and are applied at the **match level** — every frame from the same match lands in the same split.
This prevents the model from seeing frames from match X in training and other frames from the same match in the test set (data leakage).

In [221]:
splits = match_splits.load_match_splits(CONFIG_PATH)

for split_name, ids in splits.items():
    print(f'  {split_name:12s}: {len(ids)} matches')

df = match_splits.add_data_split_column(df, splits, match_col='t.match_id')

print()
print('Frames per split:')
display(
    df.groupby('data_split').agg(
        frames=('t.match_id', 'count'),
        matches=('t.match_id', 'nunique'),
    ).reset_index()
)

  split_strategy: 26 matches
  train       : 23 matches
  validation  : 5 matches
  test        : 5 matches

Frames per split:


,data_split,frames,matches
0,test,292456,5
1,train,1397290,23
2,validation,296884,5


## 3. Feature Engineering

Two features are computed from the tracking data and added to every row. The binary target `is_pass` is derived directly from `p.event_label` — no additional logic is needed.

### 3.1 Ball speed (rolling ±5 frames)

`ball_speed_avg_xy` is computed for every row as the rolling mean of frame-to-frame 2D ball displacement, using the 5 frames before and 5 frames after the current frame.

For each consecutive pair of frames `(i, i+1)`:

```
step = sqrt((ball_x[i+1] - ball_x[i])^2 + (ball_y[i+1] - ball_y[i])^2)
```

Each row's `ball_speed_avg_xy` is the mean of the steps in an 11-frame rolling window centered on that row. The window never crosses `(match_id, period)` boundaries. Steps where either frame's ball position is `NaN` are excluded from the mean. If no valid step exists in the window, the value is `NaN`.

Unit: **metres per frame**. Multiply by 10 for m/s.

In [222]:
# Find a period that has some valid ball coordinates to illustrate the rolling calculation
ball_valid = df[df['t.ball_x'].notna() & df['t.ball_y'].notna()]
if len(ball_valid) > 0:
    sample_match = str(ball_valid['t.match_id'].iloc[0])
    sample_period = ball_valid.loc[
        ball_valid['t.match_id'] == sample_match, 't.period'
    ].iloc[0]
else:
    sample_match = str(df['t.match_id'].iloc[0])
    sample_period = df.loc[df['t.match_id'] == sample_match, 't.period'].iloc[0]

period_df = (
    df[(df['t.match_id'] == sample_match) & (df['t.period'] == sample_period)]
    .sort_values('t.frame')
    .reset_index(drop=True)
)

bx = pd.to_numeric(period_df['t.ball_x'], errors='coerce')
by = pd.to_numeric(period_df['t.ball_y'], errors='coerce')
step = np.sqrt(bx.diff() ** 2 + by.diff() ** 2)
rolling_speed = step.rolling(window=11, center=True, min_periods=1).mean()

demo = period_df[['t.frame', 't.Videotimestamp', 't.ball_x', 't.ball_y']].copy()
demo['frame_step_xy'] = step.values
demo['ball_speed_avg_xy'] = rolling_speed.values

print(f'Match {sample_match}, period {sample_period}  ({len(period_df):,} frames)')
print('First 15 rows — rolling speed calculation:')
display(demo.head(15).round(4))

valid_steps = step.dropna()
print(f'\nValid frame steps in this period : {len(valid_steps):,}')
if len(valid_steps) > 0:
    print(f'Mean step                        : {valid_steps.mean():.4f} m/frame')

Match 678949, period 1  (31,658 frames)
First 15 rows — rolling speed calculation:


,t.frame,t.Videotimestamp,t.ball_x,t.ball_y,frame_step_xy,ball_speed_avg_xy
0,0,1.0,NaN,NaN,NaN,NaN
1,1,1.1,NaN,NaN,NaN,NaN
2,2,1.2,NaN,NaN,NaN,NaN
3,3,1.3,NaN,NaN,NaN,NaN
4,4,1.4,NaN,NaN,NaN,NaN
5,5,1.5,NaN,NaN,NaN,NaN
6,6,1.6,NaN,NaN,NaN,NaN
7,7,1.7,NaN,NaN,NaN,NaN
8,8,1.8,NaN,NaN,NaN,NaN
9,9,1.9,NaN,NaN,NaN,NaN



Valid frame steps in this period : 13,422
Mean step                        : 4.1491 m/frame


### 3.2 Closest player to the ball

`closest_player_id` and `closest_player_team_id` are computed for every row independently — there is no concept of a "primary event frame".

**Steps:**
1. Take ball `(x, y)` from the row. If either coordinate is missing → both outputs are `NaN`.
2. Loop over all player slots in the tracking data.
3. Skip any slot where `visible != True` or player coordinates are missing.
4. Compute the 2D Euclidean distance from each eligible player to the ball.
5. The slot with the smallest distance wins — its `player_id` and `team_id` become `closest_player_id` and `closest_player_team_id`.
6. If no eligible player exists → both outputs are `NaN`.

Implemented as a vectorized `(n_rows × n_slots)` distance matrix over all rows at once.

In [223]:
import re as _re

# Find rows in the sample period that have valid ball coordinates
rows_with_ball = period_df[
    period_df['t.ball_x'].notna() & period_df['t.ball_y'].notna()
].reset_index(drop=True)

if len(rows_with_ball) == 0:
    print('No rows with valid ball coordinates in this period — '
          'closest_player outputs will be NaN for all rows here.')
else:
    example_row = rows_with_ball.iloc[0]

    ball_x = pd.to_numeric(example_row['t.ball_x'], errors='coerce')
    ball_y = pd.to_numeric(example_row['t.ball_y'], errors='coerce')

    print(f'Frame {int(example_row["t.frame"])}  (timestamp {example_row["t.Videotimestamp"]})')
    print(f'Ball position : ({ball_x:.2f}, {ball_y:.2f})')
    print(f'Label         : {example_row["p.event_label"]}')

    player_x_cols = [c for c in df.columns if _re.match(r't\.player_\d+_x$', c)]
    slots = sorted(_re.search(r't\.player_(\d+)_x', c).group(1) for c in player_x_cols)

    candidates = []
    for slot in slots:
        vis_raw = example_row.get(f't.player_{slot}_visible')
        visible = bool(vis_raw) if isinstance(vis_raw, bool) else str(vis_raw).lower() == 'true'
        if not visible:
            continue
        px = pd.to_numeric(example_row.get(f't.player_{slot}_x'), errors='coerce')
        py = pd.to_numeric(example_row.get(f't.player_{slot}_y'), errors='coerce')
        if pd.isna(px) or pd.isna(py):
            continue
        dist_m = float(np.sqrt((px - ball_x) ** 2 + (py - ball_y) ** 2))
        candidates.append({
            'slot'      : slot,
            'player_id' : example_row.get(f't.player_{slot}_id'),
            'team_id'   : example_row.get(f't.player_{slot}_team_id'),
            'dist_m'    : round(dist_m, 3),
        })

    if not candidates:
        print('\nNo visible players → closest_player_id = NaN, closest_player_team_id = NaN')
    else:
        cdf = pd.DataFrame(candidates).sort_values('dist_m').reset_index(drop=True)
        print(f'\nVisible players: {len(cdf)}.  Five closest:')
        display(cdf.head(5))
        winner = cdf.iloc[0]
        print(f'\nclosest_player_id      = {winner["player_id"]}')
        print(f'closest_player_team_id = {winner["team_id"]}')
        print(f'distance to ball       = {winner["dist_m"]} m')

Frame 26  (timestamp 3.52)
Ball position : (43.59, 42.51)
Label         : no event

No visible players → closest_player_id = NaN, closest_player_team_id = NaN


## 4. Preview

The training tables are loaded from the split parquets written by the script.

In [230]:
display(train.tail(5))

,t.match_id,t.match_clock,t.frame,t.Videotimestamp,t.period,t.cam,t.ball_x,t.ball_y,t.ball_z,t.player_count,t.visible_player_count,t.player_01_team_id,t.player_01_id,t.player_01_x,t.player_01_y,t.player_01_visible,t.player_02_team_id,t.player_02_id,t.player_02_x,t.player_02_y,t.player_02_visible,t.player_03_team_id,t.player_03_id,t.player_03_x,t.player_03_y,t.player_03_visible,t.player_04_team_id,t.player_04_id,t.player_04_x,t.player_04_y,t.player_04_visible,t.player_05_team_id,t.player_05_id,t.player_05_x,t.player_05_y,t.player_05_visible,t.player_06_team_id,t.player_06_id,t.player_06_x,t.player_06_y,t.player_06_visible,t.player_07_team_id,t.player_07_id,t.player_07_x,t.player_07_y,t.player_07_visible,t.player_08_team_id,t.player_08_id,t.player_08_x,t.player_08_y,t.player_08_visible,t.player_09_team_id,t.player_09_id,t.player_09_x,t.player_09_y,t.player_09_visible,t.player_10_team_id,t.player_10_id,t.player_10_x,t.player_10_y,t.player_10_visible,t.player_11_team_id,t.player_11_id,t.player_11_x,t.player_11_y,t.player_11_visible,t.player_12_team_id,t.player_12_id,t.player_12_x,t.player_12_y,t.player_12_visible,t.player_13_team_id,t.player_13_id,t.player_13_x,t.player_13_y,t.player_13_visible,t.player_14_team_id,t.player_14_id,t.player_14_x,t.player_14_y,t.player_14_visible,t.player_15_team_id,t.player_15_id,t.player_15_x,t.player_15_y,t.player_15_visible,t.player_16_team_id,t.player_16_id,t.player_16_x,t.player_16_y,t.player_16_visible,t.player_17_team_id,t.player_17_id,t.player_17_x,t.player_17_y,t.player_17_visible,t.player_18_team_id,t.player_18_id,t.player_18_x,t.player_18_y,t.player_18_visible,t.player_19_team_id,t.player_19_id,t.player_19_x,t.player_19_y,t.player_19_visible,t.player_20_team_id,t.player_20_id,t.player_20_x,t.player_20_y,t.player_20_visible,t.player_21_team_id,t.player_21_id,t.player_21_x,t.player_21_y,t.player_21_visible,t.player_22_team_id,t.player_22_id,t.player_22_x,t.player_22_y,t.player_22_visible,p.event_label,data_split,is_pass,ball_speed_avg_xy,closest_player_id,closest_player_team_id
1397285,689340,"[100,13]",60361,6013.7,2,"[[73.09,187.55],[-39.77,169.01],[48.89,77.24],...",58.76,41.02,2.232,22,17,7193,1211911.0,95.32,33.68,False,7193,1159693.0,78.73,24.71,False,7193,43409.0,78.09,35.47,False,7193,1547809.0,76.40,42.30,False,7193,138414.0,59.62,14.58,True,7193,1351854.0,53.86,44.90,True,7193,1290555.0,46.61,21.66,True,7193,1678283.0,66.20,7.94,True,7193,1248088.0,55.94,32.65,True,7193,1677793.0,49.34,63.60,True,7193,138216.0,68.05,39.44,True,7198,1366912.0,74.13,42.86,True,7198,1334305.0,49.45,15.89,True,7198,138284.0,65.36,34.52,True,7198,1697457.0,61.10,45.50,True,7198,1968181.0,48.21,42.59,True,7198,1211833.0,47.50,33.93,True,7198,1174081.0,48.28,22.72,True,7198,1746886.0,71.16,22.19,True,7198,1165527.0,13.30,34.11,False,7198,1302748.0,46.44,53.04,True,7198,1296758.0,56.08,41.12,True,no event,train,0,11.995429,1296758.0,7198
1397286,689340,"[100,13]",60362,6013.8,2,"[[74.19,190.36],[-40.58,171.34],[48.83,77.42],...",63.94,56.52,2.887,22,16,7193,1211911.0,95.41,33.79,False,7193,1159693.0,77.57,25.30,False,7193,43409.0,77.72,35.42,False,7193,1547809.0,76.56,42.16,False,7193,138414.0,59.53,14.59,True,7193,1351854.0,53.63,44.98,True,7193,1290555.0,46.25,22.12,True,7193,1678283.0,66.02,7.84,True,7193,1248088.0,55.80,32.77,True,7193,1677793.0,50.02,63.27,False,7193,138216.0,67.93,39.30,True,7198,1366912.0,73.88,42.85,True,7198,1334305.0,49.31,15.98,True,7198,138284.0,65.02,34.81,True,7198,1697457.0,60.86,45.31,True,7198,1968181.0,48.00,42.53,True,7198,1211833.0,47.31,34.09,True,7198,1174081.0,48.03,22.79,True,7198,1746886.0,70.87,22.09,True,7198,1165527.0,13.48,34.46,False,7198,1302748.0,46.04,53.07,True,7198,1296758.0,55.86,41.00,True,no event,train,0,11.995429,1697457.0,7198
1397287,689340,"[100,13]",60363,6013.9,2,"[[76.31,198.85],[-42.02,176.9],[48.62,77.68],[...",56.87,40.12,3.541,22,16,7193,1211911.0,95.34,33.64,False,7193,1159693.0,77.37,25.02,False,7193,43409.0,77.62,35.16,False,7193,1547809

In [231]:
SPLIT_PATHS = {
    split: MODEL_BASE_DATA_DIR / f'training_table_{split}.parquet'
    for split in ['train', 'validation', 'test']
}

for split_name, path in SPLIT_PATHS.items():
    assert path.exists(), (
        f'Missing {path.name}.\n'
        'Run: python -m driblab.features.training_table'
    )

splits_data = {name: pd.read_parquet(path) for name, path in SPLIT_PATHS.items()}
train = splits_data['train']

print('Split sizes:')
for split_name, sdf in splits_data.items():
    n_pass = int(sdf['is_pass'].sum())
    n_total = len(sdf)
    print(f'  {split_name:12s}: {n_total:,} rows  |  PASS: {n_pass:,} ({n_pass/n_total*100:.1f}%)')

print()
print(f'Columns: {train.shape[1]}  '
      f'({sum(c.startswith("t.") for c in train.columns)} t.*  +  '
      f'{sum(not c.startswith("t.") for c in train.columns)} added)')

print()
print('Event label distribution (train):')
display(
    train['p.event_label'].value_counts()
    .rename('rows')
    .to_frame()
    .assign(pct=lambda x: (x['rows'] / len(train) * 100).round(2))
)

Split sizes:
  train       : 1,397,290 rows  |  PASS: 392,905 (28.1%)
  validation  : 296,884 rows  |  PASS: 89,345 (30.1%)
  test        : 292,456 rows  |  PASS: 94,217 (32.2%)

Columns: 127  (121 t.*  +  6 added)

Event label distribution (train):


,rows,pct
p.event_label,,
no event,885293,63.36
PASS,392905,28.12
BALL TOUCH,51250,3.67
TACKLE,17967,1.29
AERIAL,13977,1.00
BALL RECOVERY,13627,0.98
TAKEON,12136,0.87
FOUL,10135,0.73


## 5. Validate

Check the output against expected criteria:
- 127 columns: 121 `t.*` + `p.event_label` + `data_split` + `is_pass` + `ball_speed_avg_xy` + `closest_player_id` + `closest_player_team_id`
- `ball_speed_avg_xy` may be `NaN` at period edges or when the ball is untracked
- `closest_player_id` / `closest_player_team_id` are `NaN` when ball position is missing or no visible player exists
- `is_pass` has no `NaN` values

In [225]:
all_rows = pd.concat(splits_data.values(), ignore_index=True)

print('=== VALIDATION ===\n')
print(f'Total rows : {len(all_rows):,}')
print(f'Columns    : {len(all_rows.columns)} (expected 127)')

non_t_cols = [c for c in all_rows.columns if not c.startswith('t.')]
print(f'Added cols : {non_t_cols}')

print('\nMissing values per added column:')
for col in non_t_cols:
    n_null = all_rows[col].isna().sum()
    pct = n_null / len(all_rows) * 100
    print(f'  {col:30s}: {n_null:,}  ({pct:.1f}%)')

print('\nData splits:')
for split_name in ['train', 'validation', 'test']:
    sdf = splits_data[split_name]
    n_pass = int(sdf['is_pass'].sum())
    n_total = len(sdf)
    print(f'  {split_name:12s}: {n_total:,} rows  |  PASS: {n_pass:,} ({n_pass/n_total*100:.1f}%)')

print('\nEvent types (train):')
for event, count in splits_data['train']['p.event_label'].value_counts().items():
    print(f'  {event}: {count:,}')

print('\nclosest_player_team_id — top values (train):')
print(splits_data['train']['closest_player_team_id'].value_counts().head(5).to_string())

=== VALIDATION ===

Total rows : 1,986,630
Columns    : 127 (expected 127)
Added cols : ['p.event_label', 'data_split', 'is_pass', 'ball_speed_avg_xy', 'closest_player_id', 'closest_player_team_id']

Missing values per added column:
  p.event_label                 : 0  (0.0%)
  data_split                    : 0  (0.0%)
  is_pass                       : 0  (0.0%)
  ball_speed_avg_xy             : 765,811  (38.5%)
  closest_player_id             : 970,119  (48.8%)
  closest_player_team_id        : 970,119  (48.8%)

Data splits:
  train       : 1,397,290 rows  |  PASS: 392,905 (28.1%)
  validation  : 296,884 rows  |  PASS: 89,345 (30.1%)
  test        : 292,456 rows  |  PASS: 94,217 (32.2%)

Event types (train):
  no event: 885,293
  PASS: 392,905
  BALL TOUCH: 51,250
  TACKLE: 17,967
  AERIAL: 13,977
  BALL RECOVERY: 13,627
  TAKEON: 12,136
  FOUL: 10,135

closest_player_team_id — top values (train):
closest_player_team_id
8561    59134
1928    37102
3664    34534
1932    32838
46      3

## 6. Output

The files were written by the script. No additional save step is needed from this notebook.

In [226]:
for split_name, path in SPLIT_PATHS.items():
    print(f'{split_name:12s}: {path.relative_to(PROJECT_ROOT)}')

train       : data/processed/model_base/training_table_train.parquet
validation  : data/processed/model_base/training_table_validation.parquet
test        : data/processed/model_base/training_table_test.parquet
